# Laboratorium 5 - rekomendacje grafowe

## Przygotowanie

 * dataset i potrzebne biblioteki s? dok?adnie takie same jak na poprzednim laboratorium
 * pobierz i wypakuj dataset: https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
   * wi?cej mo?esz poczyta? tutaj: https://grouplens.org/datasets/movielens/
 * [opcjonalnie] Utw?rz wirtualne ?rodowisko
 `python3 -m venv ./recsyslab5`
 * zainstaluj potrzebne biblioteki:
 `pip install numpy pandas sklearn gensim==3.8.3`

## Cz??? 1. - przygotowanie danych

In [ ]:
# importujemy wszystkie potrzebne pakiety

import math
import random
import numpy as np
import pandas

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split, KFold

In [ ]:
SCORE_THRESHOLD = 4.0 # recenzje z co najmniej taka ocena wezmiemy pod uwage
VECTOR_SIZE = 20 # jak dlugie powinny byc wektory osadzen wierzcholkow
NEIGHBOURS_WINDOW = 11 # tylu sasiadow wezmiemy pod uwage w algorytmie Word2Vec (symetrycznie i wliczajac biezacy element)
PATH_LENGTH = 30 # dlugosc pojedynczej losowej sciezki
PATHS_COUNT_PER_NODE = 20 # liczba losowych sciezek zaczynajacych sie w kazdym z wierzcholkow

In [ ]:
# wczytujemy oceny uytkownikow

ratings = pandas.read_csv('ratings.csv').drop(columns=['timestamp'])
ratings = ratings.where(ratings['rating'] >= SCORE_THRESHOLD).dropna()
# rozszerzamy ID tak, by sie nie powtarzaly
ratings['userId'] = ratings['userId'].apply(lambda x: 'u_' + str(int(x)))
ratings['movieId'] = ratings['movieId'].apply(lambda x: 'm_' + str(int(x)))
ratings

In [ ]:
# wczytujemy gatunki filmow

movies = pandas.read_csv('movies.csv').drop(columns=['title'])
movies['movieId'] = movies['movieId'].apply(lambda x: 'm_' + str(int(x)))
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))
movies_to_genres = movies.explode('genres')
movies_to_genres['genres'] = movies_to_genres['genres'].apply(lambda x: 'g_' + x.lower())
movies_to_genres = movies_to_genres.rename(columns = {'genres': 'genre'})
movies_to_genres

In [ ]:
users = ratings['userId'].unique()
movies = movies_to_genres['movieId'].unique()
genres = movies_to_genres['genre'].unique()
print(genres)

## Cz??? 2. - spacer po grafie

In [ ]:
from collections import defaultdict
from random import random, choice
import random

def generate_walks(ratings, movies_to_genres, paths_per_node, path_length):
    paths = []
    memory = defaultdict(list)
    for idx, node in enumerate(np.concatenate([users, movies, genres])):
        for _ in range(PATHS_COUNT_PER_NODE):
            path = []
            for _ in range(PATH_LENGTH):
                if node[0] == 'm':
                    if memory[node]:
                        sample = random.choice(memory[node])
                    else:
                        m_to_g = movies_to_genres.loc[movies_to_genres['movieId'] == node, 'genre'].to_list()
                        m_to_u = ratings.loc[ratings['movieId'] == node, 'userId'].to_list()
                        u_g = m_to_g + m_to_u
                        sample = random.choice(u_g)
                        memory[node] = u_g
                elif node[0] == 'u':
                    if memory[node]:
                        sample = random.choice(memory[node])
                    else:
                        m = ratings.loc[ratings['userId'] == node, 'movieId'].to_list()
                        sample = random.choice(m)
                        memory[node] = m
                else:
                    if memory[node]:
                        sample = random.choice(memory[node])
                    else:
                        g = movies_to_genres.loc[movies_to_genres['genre'] == node, 'movieId'].to_list()
                        sample = random.choice(g)
                        memory[node] = g
                    
                path.append(sample)
                node = sample
                    
            paths.append(path)
            
        if idx % 100 == 0:
            print(idx)
            
        
    return paths
    
walks = generate_walks(ratings, movies_to_genres, PATHS_COUNT_PER_NODE, PATH_LENGTH)

In [ ]:
# generujemy losowe sciezki w grafie
#   krawedzie reprezentowane sa w dwoch macierzach - ratings i movies
#   w wersji podstawowej wszystkie krawedzie traktujemy jako niewazone i nieskierowane
#   mozliwe ulepszenia:
#    - rozwazenie krawedzi skierowanych
#    - uwzglednienie wag krawedzi (ocen uzytkownikow)
#    - jakas forma normalizacji - obnizenia wag wierzcholkow o wysokich stopniach
#    - Node2Vec - parametry P i Q
# wynikiem powinna byc lista list - kazda z tych list zawiera kolejne ID wierzcholkow na sciezce
from enum import Enum
from random import random, choice

class Prev(Enum):
  User = 1
  Movie = 2
  Genre = 3

def my_where(arr, axis, to_find):
    a = None
    for val in arr:
      # print(val, val[axis], to_find, val[axis] == to_find)
      # print(a)
      if val[axis] == to_find:
        if a is not None:
          a = np.append(a, np.array([val]), axis=0)
        else:
          a = np.array([val])
    # print(a)
    return a

def get_paths(node_name, prev, ratings, movies_to_genres, paths_per_node, path_length):
    paths = []
    for path_number in range(paths_per_node):
      path = [node_name]  
      for path_step in range(path_length):
        # print(f"nodename_main: {node_name}")
        if prev == Prev.User:
          print(f"nodename_user: {node_name}")
          # print(ratings[:,0])
          # possible_nodes = my_where(ratings, 0, node_name)
          possible_nodes = np.where(ratings[:,0] == node_name)
          # print(ratings[possible_nodes])
          # next_node = choice(possible_nodes)[1]
          next_node = choice(ratings[possible_nodes])[1]
          prev = Prev.Movie
        elif prev == Prev.Movie:
          print(f"nodename_movie: {node_name}")
          if random() < .5: #go to user
            # possible_nodes = my_where(ratings, 1, node_name)
            possible_nodes = np.where(ratings[:,1] == node_name)
            # print(ratings[possible_nodes])
            # print(choice(ratings[possible_nodes])[0])
            # next_node = choice(possible_nodes)[0]
            next_node = choice(ratings[possible_nodes])[0]
            prev = Prev.User
          else: #go to genre
            # possible_nodes = my_where(movies_to_genres, 0, node_name)
            possible_nodes = np.where(movies_to_genres[:,0] == node_name)
            next_node = choice(movies_to_genres[possible_nodes])[1]
            prev = Prev.Genre
        elif prev == Prev.Genre:
          print(f"nodename_genre: {node_name}")
          # possible_nodes = my_where(movies_to_genres, 1, node_name)
          possible_nodes = np.where(movies_to_genres[:,1] == node_name)
          next_node = choice(movies_to_genres[possible_nodes])[0]
          prev = Prev.Movie        
        node_name = next_node
        path.append(next_node)
      paths.append(path)
    return paths



def generate_walks(ratings, movies_to_genres, paths_per_node, path_length):
    paths = []
    # print(f"ratings: \n{ratings}")
    # print(f"movies to genres: \n{movies_to_genres}")
    # print(f"pathspernode: \n{paths_per_node}")
    # print(f"path_length: \n{path_length}")
    # print(f"unique users: {ratings['userId'].unique()}")
    # print(f"unique movies: {ratings['movieId'].unique()}")
    # print(f"unique genres: {movies_to_genres['genre'].unique()}")
    ratings = ratings.to_numpy()
    movies_to_genres = movies_to_genres.to_numpy()
    
    possible_nodes = np.where(ratings[:,1] == 'm_108090')
    print(ratings[possible_nodes])
    for i in ratings:
      if i[1] == 'm_108090':
        print(i)
    for user_id in users:
      path = get_paths(user_id, Prev.User, ratings, movies_to_genres, paths_per_node, path_length)
      paths.extend(path)
    for movie_id in movies:
      path = get_paths(movie_id, Prev.Movie, ratings, movies_to_genres, paths_per_node, path_length)
      paths.extend(path)
    for genre in genres:
      path = get_paths(genre, Prev.Genre, ratings, movies_to_genres, paths_per_node, path_length)
      paths.extend(path)

    # ...
    return paths
    
walks = generate_walks(ratings, movies_to_genres, PATHS_COUNT_PER_NODE, PATH_LENGTH)

## Cz??? 3. - obliczenie osadze?

In [ ]:
# trenujemy model
#   zauwaz, ze wszystkie trzy rodzaje wierzcholkow beda reprezentowane tak samo, w tej samej przestrzeni

model = Word2Vec(sentences=walks, size=VECTOR_SIZE, window=NEIGHBOURS_WINDOW, min_count=1, workers=4)
embeddings = model.wv

## Cz??? 4. - rekomendacje i zastosowania

In [ ]:
PULP_FICTION = 'm_296'
TOY_STORY = 'm_1'
PLANET_OF_THE_APES = 'm_2529'

In [ ]:
# wyszukajmy K najpodobniejszych film?w do danego
# porownaj wyniki dla odleglosci euklidesowej i cosinuswej, np. na trzech powyzszych filmach
from math import sqrt

def euclidian_distance(i, j):
    return sqrt(sum(np.square(i-j)))

def cosine_distance(i, j):
    return -1 * sum(np.multiply(i, j)) / sqrt(sum(np.square(i)) * sum(np.square(j)))

def k_most_similar_movies(movie_id, K, embeddings, distance_fun):
    # ...
    similars = sorted([(movie, distance_fun(embeddings[movie_id], embeddings[movie])) for movie in movies], key=lambda x: x[1])
    return similars[1:K+1]

def k_most_similar_default(id, K, embeddings):
  return embeddings.most_similar(id, topn=K)

print(k_most_similar_default(PULP_FICTION, 5, embeddings))
print(k_most_similar_movies(PULP_FICTION, 5, embeddings, cosine_distance))
print(k_most_similar_movies(PULP_FICTION, 5, embeddings, euclidian_distance))

In [ ]:
# wyszukajmy k filmow najblizszych uzytkownikowi
# wykorzystaj funkcje z poprzedniej komorki

def k_best_movies_for_user(user_id, K, embeddings, distance_fun):
    # ...
    similars = sorted([(movie, distance_fun(embeddings[user_id], embeddings[movie])) for movie in movies], key=lambda x: x[1])
    return similars[:K]

print(k_best_movies_for_user('u_1', 10, embeddings, cosine_distance))
print(k_best_movies_for_user('u_1', 10, embeddings, euclidian_distance))
# print(k_most_similar_movies_default('u_1', 10, embeddings,))

In [ ]:
genre_to_movies = {}
movies_to_genres_np = movies_to_genres.to_numpy()
for movie in movies:
  genres2 = []
  for item in movies_to_genres_np:
    if item[0] == movie:
      genres2.append(item[1])
  # genres = movies_to_genres_np[np.where(movies_to_genres_np[0]==movie)]
  # print(movies_to_genres_np)
  # print(movie)
  # print(movies_to_genres_np[np.where(movies_to_genres_np[0]==movie)])
  for genre in genres2:
    try:
      genre_to_movies[genre].append(movie)
    except:
      genre_to_movies[genre] = [movie]
print(genre_to_movies)

In [ ]:
# sprobujmy czegos bardziej skomplikowanego
#   znajdz ulubiony gatunek filmowy uzytkownika
#   a nastepnie zaproponuj K filmow z tego gatunku - ale nie tych najblizszych uzytkownikowi
#   (zaproponuj, w jaki sposob dobrac filmy interesujace, ale nie z najblizszego otoczenia)

def k_from_favourite_genre(user_id, K, embeddings, distance_fun):
    # ...
    fav_genre = sorted([(genre, distance_fun(embeddings[user_id], embeddings[genre])) for genre in genres], key=lambda x: x[1])[0][0]
    movies_from_genre = genre_to_movies[fav_genre]
    sorted_movies = sorted([(movie, distance_fun(embeddings[user_id], embeddings[movie])) for movie in movies_from_genre], key=lambda x: x[1])[:5*K]
    proposed_movies = set()
    while len(proposed_movies) < K:
      proposed_movies.add(choice(sorted_movies))
    return list(proposed_movies)

    # return k_from_genre
  
k_from_favourite_genre('u_100', 5, embeddings, cosine_distance)

In [ ]:
# Na koniec najbardziej skomplikowany algorytm - odpowiednik "Radia utworu" w Spotify.
#   Zaczynamy od jednego filmu, a nastepnie wyznaczamy kolejne, wedrujac po przestrzeni, w ktorej wszystkie elementy sa osadzone.
#   Zaproponuj, jak zdefiniowac podzbior filmow, z ktorych bedziemy wybierac (np. filmy odlegle o min. a i max. b od danego)
#   oraz jak generowac kolejny skok (tak, zeby seria rekomendacji nie byla zbyt monotonna, ale rownoczesnie zgodna z gustem uzytkownika)

# B?d? trzy przypadki, 
#     1: losuje nast?pny film kt?ry jest podobny do danego filmu
#     2: losuje film ulubionej kategorii (upgrade: wybieram podobn? kategori? dla danego filmu i z tych film?w wybieram rekomendacj? )
#     3: losuj? film kt?ry spodoba? si? u?ytkownikowi kt?ry jest podobny do u?ytkownika g??wnego

def get_playlist(start_movie_id, user_id, K, embeddings):
    playlist = set()
    dist_func = cosine_distance
    movie = start_movie_id
    while len(playlist) < K:
        randed = random.random()
        if randed < .3333333: #1
          movie = choice(k_most_similar_movies(movie, 5, embeddings, dist_func))[0]
        elif randed < .6666666: #2
          movie = choice(k_from_favourite_genre(user_id, 5, embeddings, dist_func))[0]
        else: #3
          similar_user = choice(k_most_similar_default(user_id, 10, embeddings))[0]
          movie = choice(k_best_movies_for_user(similar_user, 10, embeddings, dist_func))[0]
        playlist.add(movie)
    # ...
    return playlist
  
get_playlist(PULP_FICTION, 'u_100', 10, embeddings)
